# 05 — .NET 11: Decimal32/64/128 and Complex<T>
**JupyterSharp** kernel running on the **.NET 11** runtime (RC2 at the time of writing): `scripts/run-notebooks.ps1` starts it with `DOTNET_ROLL_FORWARD=LatestMajor` and `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`. Roslyn scripting compiles against the assemblies of the running runtime, so the new types are visible without recompiling the kernel. The numerical libraries MathemaSharp.Core, TimeSeries and Quantum are net11.0 and generic over these same formats; this notebook uses the runtime types directly, without loading them.

IEEE decimal: 7/16/34 significant digits, 32/64/128-bit formats; NaN, infinity and signed zero. `System.Decimal` uses 128 bits of storage but a 96-bit coefficient, 28–29 digits, scale 0–28 and no NaN/infinity: it is NOT Decimal128.
Sources: [overview](https://learn.microsoft.com/en-us/dotnet/core/whats-new/dotnet-11/overview), [Decimal32](https://learn.microsoft.com/en-us/dotnet/api/system.numerics.decimal32?view=net-11.0), [Complex<T>](https://learn.microsoft.com/en-us/dotnet/api/system.numerics.complex-1?view=net-11.0).

In [ ]:
#load "../shared/Kernel.csx"
using System.Numerics;
using System.Globalization;
using System.Runtime.InteropServices;
Console.WriteLine(RuntimeInformation.FrameworkDescription);
if (Environment.Version.Major < 11) throw new InvalidOperationException("The .NET 11 runtime is required: run with scripts/run-notebooks.ps1 or set DOTNET_ROLL_FORWARD.");

.NET 11.0.0-rc.3.26522.148

In [ ]:
var culture = CultureInfo.InvariantCulture;
var a32 = Decimal32.Parse("0.1",culture);
var b32 = Decimal32.Parse("0.2",culture);
Console.WriteLine(a32+b32 == Decimal32.Parse("0.3",culture));
Console.WriteLine(Decimal64.Parse("1",culture)/Decimal64.Parse("3",culture));
Console.WriteLine(Decimal128.Parse("1",culture)/Decimal128.Parse("3",culture));
Console.WriteLine(Decimal32.One/Decimal32.Zero);
Console.WriteLine(Decimal32.IsNaN(Decimal32.Zero/Decimal32.Zero));
Console.WriteLine(Decimal32.Sqrt(Decimal32.Parse("2",culture)));
Console.WriteLine(Decimal32.Sin(Decimal32.Pi));

True
0.3333333333333333
0.3333333333333333333333333333333333
Infinity
True
1.414214
-3.464102E-07

In [ ]:
T Sum11<T>(T[] values) where T : INumberBase<T>
{
    var total = T.Zero;
    foreach (var value in values) total += value;
    return total;
}
Console.WriteLine(Sum11(new[] {a32,b32}));
var zf = new Complex<float>(3f,4f);
Console.WriteLine(zf.GetMagnitude());
var zd = new Complex<Decimal64>(Decimal64.Parse("3",culture),Decimal64.Parse("4",culture));
Console.WriteLine(zd.GetMagnitude());
Console.WriteLine(Sum11(new[] {zd,zd}));

0.3
5
5.000000000000000
<6; 8>

In [ ]:
var source = "12.5;rest";
if (Decimal32.TryParsePartial(source,NumberStyles.Float,culture,out var value,out var consumed))
    Console.WriteLine($"value={value}; consumed={consumed}; rest={source.Substring(consumed)}");

value=12.5; consumed=4; rest=;rest

## Experiments and limits
Parse 12345678 with Decimal32, Decimal64 and Decimal128: tell significant digits from decimal places. Try `BitIncrement(One)`, `Quantize` and checked conversions. The new formats expose transcendental functions and IFloatingPointIeee754; `decimal` does not. Complex<T> requires IFloatingPointIeee754<T> and IMinMaxValue<T>: Complex<decimal> is not valid.
## .NET 11 benchmark plan (not yet implemented)
Repeat the same Sum11 kernel with Decimal32/64/128 arrays pre-parsed in GlobalSetup and compare with decimal/double; measure Parse, Sqrt and Complex<float>/Complex<Decimal64> separately. Do not include parsing in the sum benchmark. No performance result on the new formats has been collected: do not assume hardware acceleration or equivalent precision.